In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('./analysis_data')
OUT = BASE / 'ncs'
OUT.mkdir(parents=True, exist_ok=True)

EPS = 1e-8

In [2]:
# Load per-story metrics already computed elsewhere (zero policy)
df_coref = pd.read_csv(BASE / 'coreference' / 'coref_metrics.csv')[['model', 'prompt', 'story_id', 'coref_ratio']].copy()
df_disc = pd.read_csv(BASE / 'implicit_connectives' / 'discourse_metrics.csv')[['model', 'prompt', 'story_id', 'discourse_diversity']].copy()
df_topic = pd.read_csv(BASE / 'topic_modelling' / 'topic_switch_metrics.csv')[['model', 'prompt', 'story_id', 'topic_switch_rate']].copy()
df_mci = pd.read_csv(BASE / 'mci' / 'mci_ratio_metrics_per_story_zero_policy.csv')[['model', 'prompt', 'story_id', 'mci_tanh_mcc_over_gv']].copy()

# Ensure story_id types align
for df in [df_coref, df_disc, df_topic, df_mci]:
    df['story_id'] = df['story_id'].astype(int)

print('Loaded rows:')
print('  coref :', len(df_coref))
print('  disc  :', len(df_disc))
print('  topic :', len(df_topic))
print('  mci   :', len(df_mci))

Loaded rows:
  coref : 720
  disc  : 720
  topic : 720
  mci   : 720


In [3]:
# Character is per-character; convert to per-story by averaging char_coherence within story
df_char_pc = pd.read_csv(BASE / 'character' / 'character_metrics_per_character_zero_policy.csv')[['model', 'prompt', 'story_id', 'char_coherence']].copy()
df_char_pc['story_id'] = df_char_pc['story_id'].astype(int)

df_char = (
    df_char_pc.groupby(['model', 'prompt', 'story_id'], as_index=False)['char_coherence']
    .mean()
)

print('character per-character rows:', len(df_char_pc))
print('character per-story rows    :', len(df_char))
df_char.head()

character per-character rows: 1453
character per-story rows    : 720


,model,prompt,story_id,char_coherence
0,claude45,large,345,0.761594
1,claude45,large,515,0.376588
2,claude45,large,721,0.253865
3,claude45,large,723,0.462117
4,claude45,large,733,0.649322


In [7]:
# Merge all five components at story level
df_ncs = (
    df_coref
    .merge(df_disc, on=['model', 'prompt', 'story_id'], how='inner')
    .merge(df_char, on=['model', 'prompt', 'story_id'], how='inner')
    .merge(df_topic, on=['model', 'prompt', 'story_id'], how='inner')
    .merge(df_mci, on=['model', 'prompt', 'story_id'], how='inner')
)

components = [
    'coref_ratio',
    'discourse_diversity',
    'char_coherence',
    'topic_switch_rate',
    'mci_tanh_mcc_over_gv'
]

# Arithmetic mean (raw values)
df_ncs['ncs_arithmetic'] = df_ncs[components].mean(axis=1)

# Geometric mean requires non-negative values
neg_counts = (df_ncs[components] < 0).sum()
print('Negative-value diagnostics (for geometric mean):')
print(neg_counts.to_string())

geo_components = df_ncs[components].clip(lower=0)
df_ncs['ncs_geometric'] = np.exp(np.log(geo_components + EPS).mean(axis=1))

print('Merged per-story NCS rows:', len(df_ncs))
print('Stories per model/prompt:')
display(df_ncs.groupby(['model', 'prompt'])['story_id'].count().reset_index(name='n_stories'))
df_ncs.head()

Negative-value diagnostics (for geometric mean):
coref_ratio             0
discourse_diversity     0
char_coherence          0
topic_switch_rate       0
mci_tanh_mcc_over_gv    5
Merged per-story NCS rows: 720
Stories per model/prompt:


,model,prompt,n_stories
0,claude45,large,60
1,claude45,original,60
2,gpt4o,large,60
3,gpt4o,original,60
4,human,large,60
5,human,original,60
6,internvl3,large,60
7,internvl3,original,60
8,llama4scout,large,60
9,llama4scout,original,60


,model,prompt,story_id,coref_ratio,discourse_diversity,char_coherence,topic_switch_rate,mci_tanh_mcc_over_gv,ncs_arithmetic,ncs_geometric
0,human,original,13683,0.998852,0.244919,0.761594,0.168355,0.649252,0.564594,0.458962
1,human,original,13596,0.616018,0.635149,0.000000,0.403438,0.000000,0.330921,0.000436
2,human,original,12423,0.828877,0.379949,0.611856,0.503741,0.954949,0.655874,0.621456
3,human,original,12358,0.996899,0.244919,0.000000,0.610594,0.000000,0.370482,0.000431
4,human,original,11719,0.994905,0.635149,0.541553,0.347288,0.997154,0.703210,0.652755


In [8]:
# Aggregate NCS by model/prompt
ncs_agg = (
    df_ncs.groupby(['model', 'prompt'])
    .agg(
        n_stories=('story_id', 'count'),
        ncs_arithmetic_mean=('ncs_arithmetic', 'mean'),
        ncs_arithmetic_std=('ncs_arithmetic', 'std'),
        ncs_geometric_mean=('ncs_geometric', 'mean'),
        ncs_geometric_std=('ncs_geometric', 'std')
    )
    .reset_index()
)

ncs_agg

,model,prompt,n_stories,ncs_arithmetic_mean,ncs_arithmetic_std,ncs_geometric_mean,ncs_geometric_std
0,claude45,large,60,0.403739,0.119659,0.259843,0.207778
1,claude45,original,60,0.412418,0.118527,0.272528,0.212176
2,gpt4o,large,60,0.456155,0.115412,0.319378,0.208545
3,gpt4o,original,60,0.455157,0.115801,0.317019,0.205164
4,human,large,60,0.466622,0.111899,0.364572,0.206558
5,human,original,60,0.500014,0.143024,0.363801,0.266219
6,internvl3,large,60,0.405709,0.134544,0.271808,0.205282
7,internvl3,original,60,0.432624,0.127074,0.304617,0.203271
8,llama4scout,large,60,0.378052,0.144570,0.171512,0.239785
9,llama4scout,original,60,0.322128,0.099867,0.059710,0.153561


In [ ]:
# ncs arithmetic mean: average coherence across 5 components, higher means better average balance overall
# ncs geometric mean: coherence sensitive to weak dimensions, higher means fewer stories where one component collapses/diverges

# high arithmetic and high geometric -> both good average and consistent cross-story coherence
# high arithmetic and much lower geometric -> average looks good, but some weak metrics/components can diverse in individual stories (helps to identify dimensions where models/humans have very deviating scores?)
# low geometric would mean that at least one metric is frequently near zero in many stories

# humans have higher arithmetic (good balance), geometric stays almost unchanged among prompts
# most models (except gpt) show both arithmetic and geometric increasing from large->original; humans do not show that same geometric increase
# gpt4o is the closest to prompt-invariant behaviour
# llama4scout is behaviouraly most different (large drop in both means on original, especially geometric)



In [9]:
# Save outputs
df_ncs.to_csv(OUT / 'ncs_per_story_zero_policy.csv', index=False)
ncs_agg.to_csv(OUT / 'ncs_aggregate_zero_policy.csv', index=False)

print('Saved:')
print(' ', OUT / 'ncs_per_story_zero_policy.csv')
print(' ', OUT / 'ncs_aggregate_zero_policy.csv')

Saved:
  analysis_data/ncs/ncs_per_story_zero_policy.csv
  analysis_data/ncs/ncs_aggregate_zero_policy.csv
